# NYC TLC data exploration

notebook ini dipakai buat kenalan sama bentuk data setelah bootstrap. fokusnya bukan eda panjang, cuma cek apakah source data kita masuk akal sebelum lanjut ke feature engineering.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

REPLAY_PATH = ROOT / "data" / "source" / "replay" / "yellow_tripdata_2025-01.csv"
ZONE_LOOKUP_PATH = ROOT / "data" / "metadata" / "taxi_zone_lookup.csv"


In [ ]:
trips = pd.read_csv(REPLAY_PATH, parse_dates=["tpep_pickup_datetime"])
zones = pd.read_csv(ZONE_LOOKUP_PATH)

trips.head()


In [ ]:
print(f"rows: {len(trips):,}")
print(f"zones: {trips['PULocationID'].nunique()}")
print(f"pickup min: {trips['tpep_pickup_datetime'].min()}")
print(f"pickup max: {trips['tpep_pickup_datetime'].max()}")

trips.dtypes


In [ ]:
trips.isna().sum()


In [ ]:
manhattan_zones = zones.loc[
    zones['Borough'].str.casefold().eq('manhattan'),
    ['LocationID', 'Zone'],
]

top_pickup_zones = (
    trips['PULocationID']
    .value_counts()
    .rename_axis('LocationID')
    .reset_index(name='pickup_count')
    .merge(manhattan_zones, on='LocationID', how='left')
    .head(10)
)

top_pickup_zones


In [ ]:
sample_day = trips.loc[
    trips['tpep_pickup_datetime'].dt.date
    == pd.Timestamp('2025-01-27').date()
].copy()

sample_day['timestamp'] = sample_day['tpep_pickup_datetime'].dt.floor('h')

hourly_sample = (
    sample_day.groupby(['timestamp', 'PULocationID'])
    .size()
    .rename('trip_count')
    .reset_index()
)

hourly_sample.head(10)
